In [1]:
##modules
#%matplotlib widget
%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf


from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")
import re
# Ahora importa la función
from print5 import print5
# import pymer4 
# from pymer4.models import lmer, compare

import pickle

import re

In [2]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")
    
        
    
with open(datadir / f"subjects_remove_{modality}.pkl", "rb") as f:
    subjects_remove = pickle.load(f)
    



📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_pat

# Tables and paths
Now, as specially in the event_preprocessing we are going to work with lots of different comparisons, we are going to create, instead of variables only, we are going to use comparisons 

Example: comparison_1= [zinnen, woorden]
so variable 1= comparison_1[0]

then we will create a for loop, to print analysis for all conditions

### Paths
ACW_path
PLE_path

### Tables and metric_individuals
#### Block and event

f"acw_results_subjects_all_{layer_script}.pickle" 
if you want the acf and the whole table   f"autocorrelation_subjects_all_{filter_name}_{layer_script}.pickle"

f"autocorrelation_subjects_all_{layer_script}.pickle"

**variables** =  acw_50_elect_all_epoch_all //  'intercept_y_elect_all_epoch_all', 'slope_elect_all_epoch_all'


*FOOOF*
f"df_fooof_subject_all_{filter_name}_fixed_{layer_script}.pickle" 

**variables**
Band Powers
- 'delta'
- 'theta'
- 'alpha'
- 'beta'
- 'gamma'

Exponent 
- exponents            




#### dynamic

*ACW*


f"dynamic_autocorrelation_results_subjects_all_{layer_script}.pickle"

f"dynamic_autocorrelation_subjects_all_{layer_script}.pickle"

this is for FOOOF table_dynamic_fooof_results_all_filt_1-40_event

**variables** = acw_50_slope_elect_all_epoch_all   // acw_50_std_elect_all_epoch_all 


*FOOOF*

for all results of table f"table_dynamic_fooof_all_{suffix_str}.pickle"

for **only slopes and std**  
f"table_dynamic_fooof_results_all_{suffix_str}.pickle"
f"table_dynamic_fooof_results_all_{filtering}_{layer_script}.pickle"


**variables**
Band Powers
- 'delta_slope'
- 'theta_slope'
- 'alpha_slope'
- 'beta_slope'
- 'gamma_slope'

Exponents
- 'exponents_std'



# Table of items

**f"all_subjects_merged_events_tsv_stimuli.csv"**

This table contains the following columns
Index(['Subject', 'Condition', 'Epoch', 'Epoch_relative', 'Event_code',
       'order_trial', 'onset', 'Sample_events', 'Sample_tsv',
       'Sample_difference', 'Sample_aligned', 'Sentence_tsv', 'Text_stimuli',
       'number_item_original', 'number_item', 'par_item', 'match_ratio'],
      dtype='object')


We are going to be interested in number_item and par_item The rest of the columns are the indexes, or were used to build the table, like Sentence_tsv, so they will be removed 

Indexes will be adjusted later to check the number of epochs. 

In [3]:
filtering=True
if filtering==True:
    lfreq=1
    hfreq=40
    filter_name= f"filt_{lfreq}-{hfreq}"
    print(f"Filtrado aplicado: {lfreq}-{hfreq} Hz")
elif filtering==False:
    print("No se ha aplicado filtrado.")

Filtrado aplicado: 1-40 Hz


In [4]:
## Variables of script


#modify this only if you want to use dynamic analysis, or static event analysis
if layer_script=="event":
    dynamic=False
    
elif layer_script=="block":
    dynamic=False # it will ALWAYS be false in block


## Variables of script
##path=analysis_path
path = ACW_path

if filtering:
    if dynamic==False:
        name_table = f"acw_results_subjects_all_{filter_name}_{layer_script}.pickle" 
    if dynamic==True:
        name_table=f"dynamic_autocorrelation_results_subjects_all_{filter_name}_{layer_script}.pickle"
else:   
    name_table = f"acw_results_subjects_all_{layer_script}.pickle" 
table_metric = pd.read_pickle(f"{path}\\{name_table}")

print(f"name table metric is {name_table}")


# Extraer sujetos del dataframe
subjects = table_metric["Subject"].unique()

# Filtrar sujetos eliminados
subjects = [s for s in subjects if s not in subjects_remove]

# Filtrar tabla también
table_metric = table_metric.query("Subject in @subjects")

# Remove them also from table
#metric_individual NAMES, you can select wich one you want depending of the ACW type, and if its dynamic or not
if dynamic==False:
    metrics= ['acw_50_elect_all_epoch_all',"acw_0_elect_all_epoch_all"]
    metric_individual='acw_50_elect_all_epoch_all'
elif dynamic==True:
    metrics= ['acw_50_slope_elect_all_epoch_all',"acw_0_slope_elect_all_epoch_all"]
    metric_individual='acw_50_slope_elect_all_epoch_all'



if '_elect_all_epoch_all' in metric_individual:
    print("yes")
    metric_individual_name = metric_individual.replace('_elect_all_epoch_all', "")
else:
    metric_individual_name = metric_individual
    
    
if layer_script=="block":
    if filtering==False:
        name_table_fooof =f"df_fooof_subject_all_fixed_{layer_script}.pickle"
    elif filtering==True:
        name_table_fooof =f"df_fooof_subject_all_{filter_name}_fixed_{layer_script}.pickle"   
             
elif layer_script=="event":
    if filtering:
        if dynamic:
            name_table_fooof=f"table_dynamic_fooof_results_all_{filter_name}_{layer_script}.pickle" 
        # this means that we DONT use dynamic, we go to static event table    
        elif dynamic==False:
            name_table_fooof =f"df_fooof_subject_all_{filter_name}_fixed_{layer_script}.pickle" 
            
    elif filtering==False:
        name_table_fooof =f"df_fooof_subject_all_fixed_{layer_script}.pickle"
    

print(f"name table fooof is {name_table_fooof}")
table_fooof = pd.read_pickle(f"{path}\\{name_table_fooof}")
table_fooof = table_fooof.query("Subject in @subjects")




if layer_script=="event":
    path_csv_table=datadir / f"all_subjects_merged_events_tsv_stimuli.csv"
    all_subjects_merged_events_tsv_stimuli=pd.read_csv(path_csv_table)
    print(f" Table all_subjects_merged_events_tsv_stimuli read")


name table metric is acw_results_subjects_all_filt_1-40_block.pickle
yes
name table fooof is df_fooof_subject_all_filt_1-40_fixed_block.pickle


# Mix tables of events and ACW
### Index of epochs  

Here i can merge the tables of  all_subjects_merged_events_tsv_stimuli  and the ACW, the problem is that they dont share **dimensionality**, because they ACW includes registrations for electrode. 

So we have to consider this the table. 

We are going to mix the tables using Subject, Condition, Epoch and Epoch_relative, to add the number of the epoch relative to EACH condition

Index of Epochs is going to be applied **ONLY IN EVENT LAYER (also in dynamic)**

In [5]:
# first we add the Epoch relative to the table
def renumber_epochs(df):
    df = df.copy()
    df["Epoch_relative"] = (
        df.groupby(["Subject", "Condition"])["Epoch"]
        .transform(lambda x: pd.factorize(x)[0])
    )
    return df




table_metric_r = renumber_epochs(table_metric)

In [6]:
if layer_script=="event":
    print(f"all_subjects_merged_events_tsv_stimuli.columns are {all_subjects_merged_events_tsv_stimuli.columns}")

print(f"table_metric_r.columns are {table_metric_r.columns}")

table_metric_r.columns are Index(['Subject', 'Condition', 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all',
       'acw_0_elect_all_epoch_all', 'Epoch_relative'],
      dtype='object')


In [7]:
if layer_script=="event":
    # now we merge both tables, 

    keys = ["Subject", "Condition", "Epoch", "Epoch_relative"]

    table_metric_with_stimuli = table_metric_r.merge(
        all_subjects_merged_events_tsv_stimuli,
        on=keys,
        how="left",        # conserva todas las filas de table_metric_r (por electrodo)
        validate="m:1"     # muchas (electrodos) -> una (evento) por clave
    )

    table_metric_with_stimuli
    
    
        # This checks the number of merges

    col_check = "Sample_events"   # columna que venga solo del df stimuli

    n_total = len(table_metric_with_stimuli)
    n_match = table_metric_with_stimuli[col_check].notna().sum()
    n_no_match = table_metric_with_stimuli[col_check].isna().sum()

    print(f"Total filas: {n_total}")
    print(f"✔️ Con merge: {n_match}")
    print(f"❌ Sin merge: {n_no_match}")
    

#now we rename them both, to maintain nomenclature
if layer_script=="event":    # we use the table with the stimuli
    table_metric_to_merge=table_metric_with_stimuli.copy()

if layer_script=="block":   # we use only metric table
    table_metric_to_merge=table_metric_r.copy()

    
    


# Mix tables of ACW and FOOOF

### Index of epochs  ACW and FOOOF
 
So I have a problem here, as FOOOF for dynamic events is using relative indexed for epochs, BUT ACW is using absolute indexes

For the LMM models, I have to know EXPLICITELY wich epoch (and item ) I am using, so check what are the indexes for

ACW
- Block
- Event
- Dynamic

FOOOF
- Block
- Event
- Dynamic



To solve this, Im going to use Epoch_relative, using **renumber_epochs** for table_fooof, although it was previously used also with table_metric_r


In [8]:
## In table fooof, epochs and epochs_relative SHOULD BE THE SAME, although i create this row to maintain nomenclature

table_fooof_r = renumber_epochs(table_fooof)

n_diff = (table_fooof_r["Epoch"] != table_fooof_r["Epoch_relative"]).sum()

# So this print is just to check, and should be =0
print("Rows where Epoch != Epoch_relative:", n_diff)

Rows where Epoch != Epoch_relative: 0


In [9]:
## this code checks differences in unique values of columns between both dataframes, table_metric_to_merge, and table_fooof_r


cols = ["Subject", "Condition", "Epoch_relative", "Elect"]


for col in cols:
    set_df = set(table_metric_to_merge[col].unique())
    set_FOOOF = set(table_fooof_r[col].unique())

    print(f"\n--- {col} ---")
    print("Solo en table_metric:", set_df - set_FOOOF)
    print("Solo en table_fooof:", set_FOOOF - set_df)


--- Subject ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Condition ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Epoch_relative ---
Solo en table_metric: set()
Solo en table_fooof: set()

--- Elect ---
Solo en table_metric: set()
Solo en table_fooof: set()


In [10]:
# Remove duplicates for safety
keys_metric = table_metric_to_merge[cols].drop_duplicates()
keys_fooof  = table_fooof_r[cols].drop_duplicates()

cols = ["Subject", "Condition", "Epoch_relative", "Elect"]

# Sort: Condition → Subject → Epoch
sort_cols = ["Condition", "Subject", "Epoch_relative"]

keys_metric = keys_metric.sort_values(sort_cols).reset_index(drop=True)
keys_fooof  = keys_fooof.sort_values(sort_cols).reset_index(drop=True)

# ---------- CHECK UNIQUE VALUES PER COLUMN ----------
print("🔎 Checking unique values per column:")
unique_ok = True

for col in cols:
    u_metric = set(keys_metric[col].unique())
    u_fooof  = set(keys_fooof[col].unique())

    if u_metric == u_fooof:
        print(f"  ✅ {col}: OK ({len(u_metric)} unique values)")
    else:
        unique_ok = False
        print(f"  ❌ {col}: DO NOT match")
        print(f"     Only in table_metric: {sorted(u_metric - u_fooof)}")
        print(f"     Only in table_fooof:  {sorted(u_fooof - u_metric)}")

# ---------- CHECK COMBINATIONS ----------
n_metric = len(keys_metric)
n_fooof = len(keys_fooof)

print(f"\nMetric rows: {n_metric}")
print(f"Fooof rows:  {n_fooof}")

if unique_ok and n_metric == n_fooof and keys_metric.merge(keys_fooof, on=cols).shape[0] == n_metric:
    print("✅ Combinations match exactly between both tables.")
else:
    print("❌ Combinations DO NOT match.")

    # Show combination differences
    only_metric = keys_metric.merge(
        keys_fooof, on=cols, how="left", indicator=True
    ).query("_merge == 'left_only'")

    only_fooof = keys_fooof.merge(
        keys_metric, on=cols, how="left", indicator=True
    ).query("_merge == 'left_only'")

    print(f"Rows only in table_metric_with_stimuli: {len(only_metric)}")
    print(f"Rows only in table_fooof:  {len(only_fooof)}")

🔎 Checking unique values per column:
  ✅ Subject: OK (94 unique values)
  ✅ Condition: OK (2 unique values)
  ✅ Epoch_relative: OK (24 unique values)
  ✅ Elect: OK (270 unique values)

Metric rows: 1149660
Fooof rows:  1149660
✅ Combinations match exactly between both tables.


## Merge table_metric_to_merge with FOOOF table

In [11]:
#now, as the merge is going to be done with epoch relative, we remove Epoch from table fooof
table_fooof_r_clean = table_fooof_r.copy().drop(columns=["Epoch"])


# And now we merge
table_merged_df = pd.merge(
    table_metric_to_merge, #table for metruc
    table_fooof_r_clean,
    on=["Subject", "Condition", "Epoch_relative", "Elect"],
    how="outer",
    indicator=True
)
## this code checks that there is no differences between both dataframes in the common columns

print(table_merged_df["_merge"].value_counts())

_merge
both          1149660
left_only           0
right_only          0
Name: count, dtype: int64


## Selection for final table

## Removal of unnecesary columns

In [12]:
print(f"table_merged_df.columns are {table_merged_df.columns} ")

table_merged_df.columns are Index(['Subject', 'Condition', 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all',
       'acw_0_elect_all_epoch_all', 'Epoch_relative', 'delta', 'theta',
       'alpha', 'beta', 'gamma', 'offsets', 'exponents', 'r2', 'error',
       '_merge'],
      dtype='object') 


In [13]:
table_merged_df

,Subject,Condition,Epoch,Elect,acw_50_elect_all_epoch_all,acw_0_elect_all_epoch_all,Epoch_relative,delta,theta,alpha,beta,gamma,offsets,exponents,r2,error,_merge
0,sub-V1001,WOORDEN,1,MLC12-4304,0.020000,0.223333,0,0.0,0.313377,0.321743,0.195515,0.0,-26.623736,1.093446,0.985273,0.038964,both
1,sub-V1001,WOORDEN,1,MLC13-4304,0.020000,0.220000,0,0.0,0.000000,0.340758,0.000000,0.0,-26.525484,1.091156,0.941521,0.092458,both
2,sub-V1001,WOORDEN,1,MLC14-4304,0.023333,0.216667,0,0.0,0.000000,0.415459,0.249343,0.0,-26.334740,1.272054,0.984859,0.047137,both
3,sub-V1001,WOORDEN,1,MLC15-4304,0.026667,0.213333,0,0.0,0.000000,0.415895,0.226756,0.0,-26.201337,1.297853,0.984957,0.046183,both
4,sub-V1001,WOORDEN,1,MLC16-4304,0.026667,0.206667,0,0.0,0.000000,0.450177,0.241165,0.0,-26.166647,1.238125,0.976374,0.059663,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1149655,sub-V1117,ZINNEN,42,MZF03-4304,0.013333,0.060000,23,0.0,0.285270,0.412646,0.215223,0.0,-27.123500,0.815780,0.984401,0.034952,both
1149656,sub-V1117,ZINNEN,42,MZO01-4304,0.013333,0.036667,23,0.0,0.000000,0.712440,0.643536,0.0,-26.498561,0.924303,0.977973,0.046381,both
1149657,sub-V1117,ZINNEN,42,MZO02-4304,0.013333,0.033333,23,0.0,0.000000,0.710358,0.499493,0.0,-26.644272,0.863191,0.983175,0.040097,both
1149658,sub-V1117,ZINNEN,42,MZO03-4304,0.013333,0.040000,23,0.0,0.000000,0.517962,0.371854,0.0,-26.994866,0.876358,0.984981,0.037204,both


In [14]:
# as the table is going to contain a HUGE AMOUNT of columns that are NOT NECESSARY for the analysis, Im not going to use them, Only for Layer_script event
if layer_script=="event":
       table_merged_df=table_merged_df.copy().drop(
              columns=['Event_code','order_trial', 'Sample_tsv',
              'Sample_difference', 'Sample_aligned', 'Sentence_tsv', 'Text_stimuli',
              'number_item_original', 'match_ratio',
              '_merge'])


       print(f"After dropping columns,table_merged_df.columns are {table_merged_df.columns} ")

In [15]:
# # # # remove this, this is just to test things

# # table_metric_avg = (
# #     table_merged_df
# #     .groupby(["Subject", "Condition", "Epoch", "Epoch_relative"], as_index=False)
# #     .mean(numeric_only=True)
# # )

# # table_metric_avg_subj= table_metric_avg[table_metric_avg["Subject"]=="sub-V1001"].copy().drop(columns=['acw_50_elect_all_epoch_all',
# #        'acw_0_elect_all_epoch_all'])
# # table_metric_avg_subj


## Remove duplicated epochs
Only in event==layer

Another thing to take into account is that **epochs are duplicated**, specifically, **the epochs for questions appear as question**, but also without it. This means that the number "epoch" is not real, which is not entirely relevant, as we are NOT going to use epoch, but par_item, but its not a clear cut. 

So Im going to remove the epochs that are duplicated with and without questions, and then indexes of epoch and epoch relative  will be corrected. 

In [16]:
if layer_script=="event":
    df = table_merged_df.copy()
    df["is_question"] = df["Condition"].astype(str).str.contains("question", case=False, na=False)

    events = df[["Subject", "Condition", "Sample_events", "is_question"]].drop_duplicates()

    events["Sample_events"] = pd.to_numeric(events["Sample_events"], errors="coerce")
    events = events.dropna(subset=["Subject", "Sample_events"])
    events["Sample_events"] = events["Sample_events"].astype("int64")

    q_all  = events[events["is_question"]][["Subject", "Sample_events"]].copy()
    nq_all = events[~events["is_question"]][["Subject", "Condition", "Sample_events"]].copy()

    matched_list = []

    for subj, nq in nq_all.groupby("Subject", sort=False):
        q = q_all[q_all["Subject"] == subj][["Sample_events"]].copy()

        if q.empty:
            continue

        nq = nq.sort_values("Sample_events").reset_index(drop=True)
        q  = q.sort_values("Sample_events").reset_index(drop=True)

        m = pd.merge_asof(
            nq,
            q.rename(columns={"Sample_events": "Sample_events_q"}),
            left_on="Sample_events",
            right_on="Sample_events_q",
            tolerance=40,
            direction="nearest"
        )

        # Ensure Subject exists (in case it gets dropped in some operations)
        m["Subject"] = subj

        matched_list.append(m)

    matched = pd.concat(matched_list, ignore_index=True) if matched_list else pd.DataFrame(
        columns=["Subject", "Condition", "Sample_events", "Sample_events_q"]
    )

    to_drop_events = matched.loc[matched["Sample_events_q"].notna(), ["Subject", "Condition", "Sample_events"]]

    print("Non-question events to drop:", len(to_drop_events))

    table_merged_cleaned_df = (
        df.merge(to_drop_events.assign(_drop=1), on=["Subject", "Condition", "Sample_events"], how="left")
        .query("_drop != 1")
        .drop(columns=["_drop", "is_question"])
    )

    print("Rows before:", len(df))
    print("Rows after:", len(table_merged_cleaned_df))

    table_merged_cleaned_df


    # ------------------------------------------------------------
    # Rebuild Epoch per Subject based on Sample_events (min sample = Epoch 0)
    # ------------------------------------------------------------

    # Create a per-event mapping (Subject + Sample_events) -> new Epoch index
    epoch_map = (
        table_merged_cleaned_df[["Subject", "Sample_events"]]
        .drop_duplicates()
        .sort_values(["Subject", "Sample_events"])
    )

    epoch_map["Epoch"] = epoch_map.groupby("Subject").cumcount()

    # Merge back so the new Epoch is replicated across all Elect rows
    table_merged_cleaned_df = table_merged_cleaned_df.drop(columns=["Epoch"], errors="ignore").merge(
        epoch_map,
        on=["Subject", "Sample_events"],
        how="left",
        validate="m:1"
    )


    #now we recalculate the indexes in Epoch_relative, to avoid possible "jumps" as we eliminated epochs here
    table_merged_cleaned_df=renumber_epochs(table_merged_cleaned_df)

    
if layer_script=="block":
    table_merged_cleaned_df = table_merged_df.copy()
    
    
table_merged_cleaned_df.head()

,Subject,Condition,Epoch,Elect,acw_50_elect_all_epoch_all,acw_0_elect_all_epoch_all,Epoch_relative,delta,theta,alpha,beta,gamma,offsets,exponents,r2,error,_merge
0,sub-V1001,WOORDEN,1,MLC12-4304,0.020000,0.223333,0,0.0,0.313377,0.321743,0.195515,0.0,-26.623736,1.093446,0.985273,0.038964,both
1,sub-V1001,WOORDEN,1,MLC13-4304,0.020000,0.220000,0,0.0,0.000000,0.340758,0.000000,0.0,-26.525484,1.091156,0.941521,0.092458,both
2,sub-V1001,WOORDEN,1,MLC14-4304,0.023333,0.216667,0,0.0,0.000000,0.415459,0.249343,0.0,-26.334740,1.272054,0.984859,0.047137,both
3,sub-V1001,WOORDEN,1,MLC15-4304,0.026667,0.213333,0,0.0,0.000000,0.415895,0.226756,0.0,-26.201337,1.297853,0.984957,0.046183,both
4,sub-V1001,WOORDEN,1,MLC16-4304,0.026667,0.206667,0,0.0,0.000000,0.450177,0.241165,0.0,-26.166647,1.238125,0.976374,0.059663,both


In [17]:
## DELETE THIS

# # if layer_script=="block":
# #     table_clean_avg = (
# #         table_merged_cleaned_df
# #         .groupby(["Subject", "Condition", "Epoch", "Epoch_relative"], as_index=False)
# #         .mean(numeric_only=True)
# #     )

# #     table_clean_avg_subj= table_clean_avg[table_clean_avg["Subject"]=="sub-V1001"].copy().drop(columns=['acw_50_elect_all_epoch_all',
# #         'acw_0_elect_all_epoch_all'])


# #     table_clean_avg_subj

### Add specific columns (only for event)

For the LMM that we will apply with R, we need to put all the columns in a certain format, in order to have all the factors, wich will be
- **Condition**: zinnen or sentence (common in both block and event layer)

From event, we rename condition to **condition_specific**, wich will contain the full name (e.g. zinnen_RC_plus_question_hit) and we decompose it in the different parts
- **Condition**: that has zinnen or wordlist
- **Condition_RC**: which contains if there is relative clause (RC) or not
- **Condition_question**: which contains if there is question, and the answer

In [18]:
#only changes in event
if layer_script == "event":

    # 1) Renombrar Condition -> Condition_specific
    table_merged_cleaned_df = table_merged_cleaned_df.rename(columns={"Condition": "Condition_specific"})

    # Asegurar Series (NO uses doble corchete aquí)
    s = table_merged_cleaned_df["Condition_specific"].astype("string")

    # 2) Nueva columna Condition (Woorden / Zinnen / <NA>)
    table_merged_cleaned_df["Condition"] = pd.NA
    mask_woorden = s.str.contains("woorden", case=False, na=False)
    mask_zinnen  = s.str.contains("zinnen",  case=False, na=False)

    table_merged_cleaned_df.loc[mask_woorden, "Condition"] = "woorden"
    table_merged_cleaned_df.loc[mask_zinnen,  "Condition"] = "zinnen"

    
    def to_Condition_RC(val):
        if pd.isna(val):
            return pd.NA

        s = str(val).lower()

        # base: woorden / zinnen
        if "woorden" in s:
            base = "woorden"
        elif "zinnen" in s:
            base = "zinnen"
        else:
            return pd.NA

        # tipo RC
        if "rc_plus" in s:
            rc = "RC_plus"
        elif "rc_neg" in s:
            rc = "RC_neg"
        else:
            return pd.NA

        return rc


    table_merged_cleaned_df["Condition_RC"] = (
        table_merged_cleaned_df["Condition_specific"]
        .apply(to_Condition_RC)
    )
    
    
    def to_condition_question(val):
        if pd.isna(val):
            return pd.NA

        s = str(val).lower()

        # base
        if "woorden" in s:
            base = "woorden"
        elif "zinnen" in s:
            base = "zinnen"
        else:
            return pd.NA

        # tipo de pregunta
        if "question_hit" in s:
            question = "hit"
        elif "question_incorrect" in s:
            question = "incorrect"
        else:
            return pd.NA

        return question

    table_merged_cleaned_df["Condition_question"] = (
        table_merged_cleaned_df["Condition_specific"]
        .apply(to_condition_question)
    )

        # 4) Reordenar columnas (dejando el resto al final)
    desired_order = [
        "Subject",
        "Condition_specific",
        "Condition",
        "Condition_RC",
        "Condition_question",
    ]

    remaining_cols = [c for c in table_merged_cleaned_df.columns if c not in desired_order]

    table_merged_cleaned_df = table_merged_cleaned_df[desired_order + remaining_cols]


# Its not necessary to do anything in block condition

In [19]:

# # table_clean_avg = (
# #     table_merged_cleaned_df
# #     .groupby(["Subject", "Condition_specific","Condition","Condition_RC", "Condition_question" ,"Epoch", "Epoch_relative"], as_index=False, dropna=False)
# #     .mean(numeric_only=True)
# # )

# # table_clean_avg_subj= table_clean_avg[table_clean_avg["Subject"]=="sub-V1001"].copy().drop(columns=['acw_50_elect_all_epoch_all',
# #     'acw_0_elect_all_epoch_all'])


# # table_clean_avg_subj

# Now we add the type of channel

In [20]:
dict_isc= pd.read_pickle(ISC_block_path /f"ISC_results_block.pkl")
dict_woorden_block=dict_isc['dict_isc_WOORDEN']
dict_zinnen_block = dict_isc['dict_isc_ZINNEN']
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted"]
print("numbers_channels_woorden", numbers_channels_woorden, "numbers_channels_zinnen", numbers_channels_zinnen)
print("len(numbers_channels_woorden)",len(numbers_channels_woorden),  "len(numbers_channels_zinnen)",len(numbers_channels_zinnen))

names_channels_woorden=dict_woorden_block["significant_channels_adjusted_names"]
names_channels_zinnen=dict_zinnen_block["significant_channels_adjusted_names"]

print("names_channels_woorden", names_channels_woorden)
print("names_channels_zinnen", names_channels_zinnen)

h_subj =0
path_epochs = epochs_clean_path / f"{subjects[h_subj]}_epochs_{layer_script}-epo.fif"
epochs = mne.read_epochs(path_epochs, preload=False)
#establecimiento de canales palabras, canales frases y canales mixtos
# Tomamos el orden original de los canales del objeto epochs
all_channels = epochs.ch_names
all_channels_numbers= [epochs.ch_names.index(ch) for ch in epochs.ch_names]


del epochs

# Palabras
numbers_channels_only_woorden = [
    ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch not in numbers_channels_zinnen
]

numbers_channels_only_zinnen = [
    ch for ch in all_channels_numbers if ch in numbers_channels_zinnen and ch not in numbers_channels_woorden
]

numbers_channels_intersection = [
    ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch in numbers_channels_zinnen
]

# Lo mismo pero usando nombres
names_channels_only_woorden = [
    ch for ch in all_channels if ch in names_channels_woorden and ch not in names_channels_zinnen
]

names_channels_only_zinnen = [
    ch for ch in all_channels if ch in names_channels_zinnen and ch not in names_channels_woorden
]

names_channels_intersection = [
    ch for ch in all_channels if ch in names_channels_woorden and ch in names_channels_zinnen
]
# ---------------------------
# Print resumen
# ---------------------------

print(f"len(significant_channels_only_woorden): {len(numbers_channels_only_woorden)}, "
      f"len(significant_channels_only_zinnen): {len(numbers_channels_only_zinnen)}, "
      f"len(significant_channels_intersection): {len(numbers_channels_intersection)}")



print("\n✅ Only woorden (names):", names_channels_only_woorden)
print("✅ Only zinnen (names):", names_channels_only_zinnen)
print("✅ Intersection (names):", names_channels_intersection)


dict_select_channels={
    "names_channels_only_woorden": names_channels_only_woorden,
    "names_channels_only_zinnen": names_channels_only_zinnen,
    "names_channels_intersection": names_channels_intersection
}

numbers_channels_woorden [  2   3   4   5   7   8   9  10  11  12  13  14  17  18  35  36  40  41
  42  43  45  46  47  48  49  51  52  53  54  56  57  58  60  61  62  64
  65  66  68  69  70  73  75  77  78  80  81  82  83  84  85  86  87  88
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 131 132 133 134 135 136 137 138 139 140 141 142 143 144 146 147
 148 149 150 151 182 183 184 188 192 196 199 200 201 202 204 206 207 209
 210 211 212 214 215 216 217 221 222 223 224 225 226 228 229 230 231 232
 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250
 251 252 253 254 255 256 257 258] numbers_channels_zinnen [  2   3   4   5   7   8   9  10  11  12  13  14  17  18  19  24  25  26
  29  30  31  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47
  48  49  51  52  53  54  55  56  57  58  59  62  66  75  77  78  79  80
  81  82  83  84  85  86  87  88  89  90 

In [21]:
# Check channel membership sets
set_woorden = set(names_channels_woorden)
set_zinnen = set(names_channels_zinnen)

def channel_labels(ch):
    """
    Returns:
      - channel_type: intersection / only_woorden / only_zinnen / NaN
      - channel_condition: woorden / zinnen / NaN
    """
    in_woorden = ch in set_woorden
    in_zinnen  = ch in set_zinnen

    # Channel_type (set relationship)
    if in_woorden and in_zinnen:
        channel_type = "channel_intersection"
    elif in_woorden:
        channel_type = "channel_only_woorden"
    elif in_zinnen:
        channel_type = "channel_only_zinnen"
    else:
        channel_type = np.nan

    # Channel_condition (base membership)
    if in_woorden:
        channel_condition = "channel_woorden"
    elif in_zinnen:
        channel_condition = "channel_zinnen"
    else:
        channel_condition = np.nan

    return channel_type, channel_condition


# Create both columns using the same function
tmp = table_merged_cleaned_df["Elect"].apply(channel_labels)

table_merged_cleaned_df["Channel_type"] = tmp.apply(lambda x: x[0])
table_merged_cleaned_df["Channel_condition"] = tmp.apply(lambda x: x[1])

table_merged_cleaned_df.head()

,Subject,Condition,Epoch,Elect,acw_50_elect_all_epoch_all,acw_0_elect_all_epoch_all,Epoch_relative,delta,theta,alpha,beta,gamma,offsets,exponents,r2,error,_merge,Channel_type,Channel_condition
0,sub-V1001,WOORDEN,1,MLC12-4304,0.020000,0.223333,0,0.0,0.313377,0.321743,0.195515,0.0,-26.623736,1.093446,0.985273,0.038964,both,NaN,NaN
1,sub-V1001,WOORDEN,1,MLC13-4304,0.020000,0.220000,0,0.0,0.000000,0.340758,0.000000,0.0,-26.525484,1.091156,0.941521,0.092458,both,NaN,NaN
2,sub-V1001,WOORDEN,1,MLC14-4304,0.023333,0.216667,0,0.0,0.000000,0.415459,0.249343,0.0,-26.334740,1.272054,0.984859,0.047137,both,channel_intersection,channel_woorden
3,sub-V1001,WOORDEN,1,MLC15-4304,0.026667,0.213333,0,0.0,0.000000,0.415895,0.226756,0.0,-26.201337,1.297853,0.984957,0.046183,both,channel_intersection,channel_woorden
4,sub-V1001,WOORDEN,1,MLC16-4304,0.026667,0.206667,0,0.0,0.000000,0.450177,0.241165,0.0,-26.166647,1.238125,0.976374,0.059663,both,channel_intersection,channel_woorden


In [22]:
#Now we want to check if the channels were added correctly

def assert_channel_sets(df, df_label, original_list, original_label):
    """
    This function checks whether the set of channels in the dataframe
    (filtered by a specific channel condition) matches exactly the set
    of channels provided in the original dictionary.

    Parameters
    ----------
    df : DataFrame
        The cleaned dataframe containing channel information.
    df_label : str
        The value used to filter the column 'Channel_type'.
    original_list : list
        The list of expected channel names from the original dictionary.
    original_label : str
        The label/name of the original dictionary (for reporting purposes).
    """

    # Extract the set of channels from the dataframe for the given condition
    df_set = set(
        df.query("Channel_type == @df_label")["Elect"]
        .dropna()
        .unique()
    )

    # Convert the expected channel list into a set
    orig_set = set(original_list)

    # Differences between dataframe channels and original dictionary channels
    only_df = df_set - orig_set
    only_orig = orig_set - df_set

    print(f"\n🔎 CHECK {df_label}")
    print(f"DataFrame count: {len(df_set)} | Original dictionary count: {len(orig_set)}")
    print("Only in dataframe:", sorted(only_df))
    print("Only in original dictionary:", sorted(only_orig))

    # Assert ensures the script stops if there is any mismatch
    assert not only_df and not only_orig, (
        f"\n❌ Mismatch in {df_label}\n"
        f"Only in dataframe: {sorted(only_df)}\n"
        f"Only in original dictionary: {sorted(only_orig)}"
    )

    print(f"✅ {df_label} matches {original_label}")


# ---------------------------------------------------
# Run channel consistency checks
# The script will stop automatically if a mismatch is detected
# ---------------------------------------------------

assert_channel_sets(
    table_merged_cleaned_df,
    "channel_only_woorden",
    names_channels_only_woorden,
    "names_channels_only_woorden"
)

assert_channel_sets(
    table_merged_cleaned_df,
    "channel_only_zinnen",
    names_channels_only_zinnen,
    "names_channels_only_zinnen"
)

assert_channel_sets(
    table_merged_cleaned_df,
    "channel_intersection",
    names_channels_intersection,
    "names_channels_intersection"
)


🔎 CHECK channel_only_woorden
DataFrame count: 13 | Original dictionary count: 13
Only in dataframe: []
Only in original dictionary: []
✅ channel_only_woorden matches names_channels_only_woorden

🔎 CHECK channel_only_zinnen
DataFrame count: 52 | Original dictionary count: 52
Only in dataframe: []
Only in original dictionary: []
✅ channel_only_zinnen matches names_channels_only_zinnen

🔎 CHECK channel_intersection
DataFrame count: 157 | Original dictionary count: 157
Only in dataframe: []
Only in original dictionary: []
✅ channel_intersection matches names_channels_intersection


# Change names from zinnen to sentence, from woorden to wordlist 
Now as we have the definite tables, we change the names 
We will also re order the columns


In [23]:
table_merged_cleaned_df.columns

Index(['Subject', 'Condition', 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all',
       'acw_0_elect_all_epoch_all', 'Epoch_relative', 'delta', 'theta',
       'alpha', 'beta', 'gamma', 'offsets', 'exponents', 'r2', 'error',
       '_merge', 'Channel_type', 'Channel_condition'],
      dtype='object')

In [24]:
table_merged_cleaned_df = table_merged_cleaned_df.replace(
    {
        "woorden": "wordlist",
        "WOORDEN": "wordlist",
        "zinnen": "sentence",
        "ZINNEN": "sentence"
    },
    regex=True
)

table_merged_cleaned_df = table_merged_cleaned_df.rename(
    columns={"Elect": "Channel"}
)

if layer_script=="event":
    # Desired column order

    if dynamic==False:

        
        desired_order = [
            "Subject",
            "Condition_specific",
            "Condition",
            "Condition_RC",
            "Condition_question",
            "Epoch",
            "Epoch_relative",
            "onset",
            "Sample_events",
            "number_item",
            "par_item",
            "Channel",
            "Channel_condition",
            "Channel_type",
            "acw_50_elect_all_epoch_all",
            "acw_0_elect_all_epoch_all",
            "delta",
            "theta",
            "alpha",
            "beta",
            "gamma",
            "offsets",
            "exponents",
            "r2",
            "error",
        ]

        # Keep any remaining columns at the end (safe version)
        remaining_cols = [c for c in table_merged_cleaned_df.columns if c not in desired_order]

        # Reorder dataframe
        table_merged_cleaned_df = table_merged_cleaned_df[desired_order + remaining_cols]
        
    elif dynamic==True:
        desired_order = [
        "Subject",
        "Condition_specific",
        "Condition",
        "Condition_RC",
        "Condition_question",
        "Epoch",
        "Epoch_relative",
        "onset",
        "Sample_events",
        "number_item",
        "par_item",
        "Channel",
        "Channel_condition",
        "Channel_type",

        # ACW
        "acw_50_slope_elect_all_epoch_all",
        "acw_50_std_elect_all_epoch_all",
        "acw_0_slope_elect_all_epoch_all",
        "acw_0_std_elect_all_epoch_all",

        # Band powers (slope + std, one by one)
        "delta_slope",
        "delta_std",
        "theta_slope",
        "theta_std",
        "alpha_slope",
        "alpha_std",
        "beta_slope",
        "beta_std",
        "gamma_slope",
        "gamma_std",

        # FOOOF params (slope + std, one by one)
        "offsets_slope",
        "offsets_std",
        "exponents_slope",
        "exponents_std"
        ]

        
        
elif layer_script == "block":

    desired_order = [
        "Subject",
        "Condition",
        "Epoch",
        "Epoch_relative",
        "Channel",
        "Channel_condition",
        "Channel_type",
        "acw_50_elect_all_epoch_all",
        "acw_0_elect_all_epoch_all",
        "delta",
        "theta",
        "alpha",
        "beta",
        "gamma",
        "offsets",
        "exponents",
        "r2",
        "error",
        "_merge",
    ]

# keep remaining columns at the end (safe version)
remaining_cols = [c for c in table_merged_cleaned_df.columns if c not in desired_order]

table_merged_cleaned_df = table_merged_cleaned_df[desired_order + remaining_cols]
    

In [25]:
table_merged_cleaned_df

,Subject,Condition,Epoch,Epoch_relative,Channel,Channel_condition,Channel_type,acw_50_elect_all_epoch_all,acw_0_elect_all_epoch_all,delta,theta,alpha,beta,gamma,offsets,exponents,r2,error,_merge
0,sub-V1001,wordlist,1,0,MLC12-4304,NaN,NaN,0.020000,0.223333,0.0,0.313377,0.321743,0.195515,0.0,-26.623736,1.093446,0.985273,0.038964,both
1,sub-V1001,wordlist,1,0,MLC13-4304,NaN,NaN,0.020000,0.220000,0.0,0.000000,0.340758,0.000000,0.0,-26.525484,1.091156,0.941521,0.092458,both
2,sub-V1001,wordlist,1,0,MLC14-4304,channel_wordlist,channel_intersection,0.023333,0.216667,0.0,0.000000,0.415459,0.249343,0.0,-26.334740,1.272054,0.984859,0.047137,both
3,sub-V1001,wordlist,1,0,MLC15-4304,channel_wordlist,channel_intersection,0.026667,0.213333,0.0,0.000000,0.415895,0.226756,0.0,-26.201337,1.297853,0.984957,0.046183,both
4,sub-V1001,wordlist,1,0,MLC16-4304,channel_wordlist,channel_intersection,0.026667,0.206667,0.0,0.000000,0.450177,0.241165,0.0,-26.166647,1.238125,0.976374,0.059663,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1149655,sub-V1117,sentence,42,23,MZF03-4304,channel_sentence,channel_only_sentence,0.013333,0.060000,0.0,0.285270,0.412646,0.215223,0.0,-27.123500,0.815780,0.984401,0.034952,both
1149656,sub-V1117,sentence,42,23,MZO01-4304,channel_sentence,channel_only_sentence,0.013333,0.036667,0.0,0.000000,0.712440,0.643536,0.0,-26.498561,0.924303,0.977973,0.046381,both
1149657,sub-V1117,sentence,42,23,MZO02-4304,NaN,NaN,0.013333,0.033333,0.0,0.000000,0.710358,0.499493,0.0,-26.644272,0.863191,0.983175,0.040097,both
1149658,sub-V1117,sentence,42,23,MZO03-4304,NaN,NaN,0.013333,0.040000,0.0,0.000000,0.517962,0.371854,0.0,-26.994866,0.876358,0.984981,0.037204,both


In [26]:
if filtering:
    output_path = ACW_path / f"table_merged_ACW_FOOOF_{filter_name}_{layer_script}.csv"
    if dynamic:
        output_path = ACW_path / f"table_merged_dynamic_ACW_FOOOF_results_{filter_name}_{layer_script}.csv"
        
else:
    output_path = ACW_path / f"table_merged_df_{layer_script}.csv"

table_merged_cleaned_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to g:\MOUS_204\MOUS_visual\output_analysis\analysis_block\acw_block\table_merged_ACW_FOOOF_filt_1-40_block.csv
